# Gene Set Activity Analysis

Scores 36 gene sets across omentum and subcutaneous adipose scRNA-seq.

**Strategy by gene set size:**

| Set size | Per-cell score | DE-based enrichment |
|---|---|---|
| ≥ 15 genes | `decoupler` ULM | `gseapy` GSEA (prerank on MAST score) |
| < 15 genes | `scanpy` `score_genes` | Mean logFC + Stouffer combined Z |

**Why `score_genes` for small sets?** ULM fits a regression and its z-scores are unreliable with few predictors.
`score_genes` computes: `mean(targets) − mean(size-matched control genes)`, which is well-defined even for 2 genes.

**Why Stouffer for small DE sets?** GSEA permutation p-values need ≥15 genes to be meaningful.
Stouffer's method weights each gene's z-score (from MAST) equally and produces a set-level z with a proper normal p-value.

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import anndata as ad
import gseapy as gp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE    = Path('/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq')
IN_DIR  = BASE / 'inputs'
OUT_DIR = BASE / 'results' / 'gene_sets'
FIG_DIR = OUT_DIR / 'figures'

for d in [OUT_DIR / 'de_enrichment', FIG_DIR / 'violin', FIG_DIR / 'umap']:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
MIN_GENES_GSEA = 15   # gene set size threshold: GSEA vs Stouffer

COHORT_PALETTE = {'Healthy': '#4393c3', 'Unhealthy': '#d6604d'}
SEX_PALETTE    = {'M': '#2166ac', 'F': '#d01c8b'}

DEPOTS = {
    'omentum': IN_DIR / 'anndatas/omentum_reannotated_finely.h5ad',
    'subq':    IN_DIR / 'anndatas/sq_reannotated_finely.h5ad',
}

DE_FILES = {
    ('omentum', 'cell_type_'):  BASE / 'results/DE/omentum/MAST_results_per_cell_type_.csv',
    ('omentum', 'cell_type2_'): BASE / 'results/DE/omentum/MAST_results_per_cell_type2_.csv',
    ('subq',    'cell_type_'):  BASE / 'results/DE/subq/MAST_results_per_cell_type_.csv',
    ('subq',    'cell_type2_'): BASE / 'results/DE/subq/MAST_results_per_cell_type2_.csv',
}

In [30]:
# adatas["subq"].obs.loc[adatas["subq"].obs["cell_type2_"] == "LECs", "cell_type2_"] = "VECs"

In [34]:
# adatas["subq"].obs["cell_type2_"] = adatas["subq"].obs["cell_type2_"].cat.remove_unused_categories()

In [36]:
# import anndata
# anndata.settings.allow_write_nullable_strings = True
# adatas["subq"].write_h5ad("/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/inputs/anndatas/sq_reannotated_finely.h5ad")

## 1. Load Data

In [9]:
adatas = {}
for name, path in DEPOTS.items():
    print(f'Loading {name}...')
    adata = sc.read_h5ad(path)
    adata.obs['depot'] = name
    adatas[name] = adata
    print(f'  {adata.n_obs} cells × {adata.n_vars} genes')
    print(f'  cell_type_:  {adata.obs["cell_type_"].nunique()} types')
    print(f'  cell_type2_: {adata.obs["cell_type2_"].nunique()} types')

Loading omentum...
  25830 cells × 26158 genes
  cell_type_:  11 types
  cell_type2_: 27 types
Loading subq...
  9127 cells × 22651 genes
  cell_type_:  8 types
  cell_type2_: 17 types


## 2. Load Gene Sets

In [10]:
gene_lists_raw = pd.read_csv(IN_DIR / 'gene_lists.csv')

gene_sets = {
    col: gene_lists_raw[col].dropna().tolist()
    for col in gene_lists_raw.columns
}

# Summarise sets — split determines DE enrichment method, not per-cell scoring
gs_summary = pd.DataFrame({
    'n_genes': {k: len(v) for k, v in gene_sets.items()},
    'de_method': {
        k: 'GSEA (prerank)' if len(v) >= MIN_GENES_GSEA else 'Stouffer Z + mean logFC'
        for k, v in gene_sets.items()
    },
}).sort_values('n_genes')

print(gs_summary.to_string())

# These splits are used only for the DE enrichment step
large_sets = {k: v for k, v in gene_sets.items() if len(v) >= MIN_GENES_GSEA}
small_sets  = {k: v for k, v in gene_sets.items() if len(v) <  MIN_GENES_GSEA}
print(f'\nLarge (≥{MIN_GENES_GSEA}, use GSEA):    {len(large_sets)} sets')
print(f'Small  (<{MIN_GENES_GSEA}, use Stouffer): {len(small_sets)} sets  → {list(small_sets.keys())}')

                                                                                         n_genes                de_method
other                                                                                          2  Stouffer Z + mean logFC
Mesencyhmal                                                                                    3  Stouffer Z + mean logFC
Adhesion                                                                                       4  Stouffer Z + mean logFC
Lipid                                                                                          6  Stouffer Z + mean logFC
GO_ENDOTHELIAL_CELL_MATRIX_ADHESION                                                            6  Stouffer Z + mean logFC
GO_LYMPHATIC_ENDOTHELIAL_CELL_DIFFERENTIATION                                                  8  Stouffer Z + mean logFC
GO_POSITIVE_REGULATION_OF_PROTEIN_GLYCOSYLATION                                                9  Stouffer Z + mean logFC
Adipogenic              

## 3. Per-Cell Scoring

All gene sets use `sc.tl.score_genes()` (Seurat-style: mean of target genes − mean of size-matched control genes).
This has a stable API and works correctly for any set size from 2 genes upward.

`ctrl_size` is set to `max(50, n_found × 10)` so the control reference is always at least 10× larger than the target set.

Scores are stored in `adata.obs` as `score_<gene_set_name>`.

In [17]:
import re
import json

def safe_name(s):
    """Column-safe gene set name."""
    return 'score_' + s.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')

score_col_map = {k: safe_name(k) for k in gene_sets}

# ── Build symbol → Ensembl / Entrez mappings from DE results ──────────────────
sym2ensembl: dict = {}
sym2entrez:  dict = {}

for path in DE_FILES.values():
    chunk = pd.read_csv(path, usecols=['primerid', 'ensembl_id', 'entrez_id'])
    for _, row in chunk.drop_duplicates('primerid').iterrows():
        sym = row['primerid']
        if pd.notna(row.get('ensembl_id')) and sym not in sym2ensembl:
            sym2ensembl[sym] = str(row['ensembl_id'])
        if pd.notna(row.get('entrez_id')) and sym not in sym2entrez:
            try:
                sym2entrez[sym] = str(int(float(row['entrez_id'])))
            except (ValueError, TypeError):
                pass

print(f'Symbol → Ensembl mappings from DE results: {len(sym2ensembl)}')
print(f'Symbol → Entrez  mappings from DE results: {len(sym2entrez)}')


def build_var_lookups(adata):
    """Build {ensembl_id: var_name} and {entrez_id: var_name} from adata.var."""
    var = adata.var.copy()
    var.index.name = 'symbol'
    var = var.reset_index()
    ensembl2var = dict(zip(var['gene_id'].dropna(), var['symbol']))
    entrez2var  = {}
    for _, row in var.dropna(subset=['entrez_id']).iterrows():
        try:
            entrez2var[str(int(float(row['entrez_id'])))] = row['symbol']
        except (ValueError, TypeError):
            pass
    return ensembl2var, entrez2var


def expand_gene_range(gene_str):
    """
    Expand 'PLIN1-5' → ['PLIN1','PLIN2','PLIN3','PLIN4','PLIN5'].
    Only triggers for LETTERS + 1-2-digit number + dash + 1-2-digit number.
    Returns [gene_str] unchanged if no range is detected.
    """
    m = re.match(r'^([A-Za-z]+)(\d{1,2})-(\d{1,2})$', gene_str.strip())
    if m:
        prefix, start, end = m.group(1), int(m.group(2)), int(m.group(3))
        if end > start:
            return [f'{prefix}{i}' for i in range(start, end + 1)]
    return [gene_str]


def is_family_prefix(gene_str):
    """
    True when the token is a pure letter string with no digits, e.g. 'CCL', 'CXCL'.
    These are intended as gene family prefixes (CCL\d+) rather than specific genes.
    """
    return bool(re.match(r'^[A-Za-z]+$', gene_str))


def expand_family_prefix(prefix, adata):
    """
    Return all adata.var_names matching ^prefix\d+$ (case-sensitive).
    E.g. 'CCL' → ['CCL1','CCL2','CCL3', ...] for whatever is in the data.
    """
    pat = re.compile(r'^' + re.escape(prefix) + r'\d+$')
    return [g for g in adata.var_names if pat.match(g)]


# alias_cache is populated by the mygene cell below; used at call-time via closure
alias_cache: dict = {}


def _lookup(sym, adata, ensembl2var, entrez2var):
    """Try to resolve one symbol to a var_name: symbol → Ensembl → Entrez."""
    if sym in adata.var_names:
        return sym, 'symbol'
    ens = sym2ensembl.get(sym)
    if ens and ens in ensembl2var:
        return ensembl2var[ens], 'ensembl'
    ent = sym2entrez.get(sym)
    if ent and ent in entrez2var:
        return entrez2var[ent], 'entrez'
    return None, None


def resolve_gene_list(genes, adata, ensembl2var, entrez2var):
    """
    Resolve a gene list to adata.var_names.

    Handling per token after range-expansion (PLIN1-5 → PLIN1…PLIN5):

    • Pure-letter tokens (CCL, CXCL, CCR, CXCR):
        → prefix expansion: all adata.var_names matching ^TOKEN\\d+$
        → no ID / alias fallback (would only return one gene)

    • Tokens with digits that fail direct match:
        1. Ensembl ID (DE-result mapping → adata.var gene_id)
        2. Entrez ID  (DE-result mapping → adata.var entrez_id)
        3. mygene canonical alias → re-run tiers 1-2 on canonical symbol

    Returns (resolved_var_names, counts_dict).
    """
    resolved = []
    seen     = set()
    counts   = {'symbol': 0, 'ensembl': 0, 'entrez': 0, 'alias': 0,
                'prefix': 0, 'missing': 0}

    for gene_raw in genes:
        for gene in expand_gene_range(gene_raw):

            # ── Family prefix (e.g. CCL, CXCL) ───────────────────────────────
            if is_family_prefix(gene) and gene not in adata.var_names:
                matches = expand_family_prefix(gene, adata)
                if matches:
                    for vn in matches:
                        if vn not in seen:
                            resolved.append(vn)
                            seen.add(vn)
                    counts['prefix'] += len(matches)
                    continue
                # nothing matched → fall through to normal tiers below

            # ── Tiers 1-3: symbol / Ensembl / Entrez ─────────────────────────
            vn, method = _lookup(gene, adata, ensembl2var, entrez2var)

            # ── Tier 4: mygene alias ──────────────────────────────────────────
            if vn is None:
                canonical = alias_cache.get(gene)
                if canonical and canonical != gene:
                    vn, _ = _lookup(canonical, adata, ensembl2var, entrez2var)
                    if vn is not None:
                        method = 'alias'

            if vn is not None:
                if vn not in seen:
                    resolved.append(vn)
                    seen.add(vn)
                counts[method or 'symbol'] += 1
            else:
                counts['missing'] += 1

    return resolved, counts

Symbol → Ensembl mappings from DE results: 11902
Symbol → Entrez  mappings from DE results: 11466


In [18]:
import mygene

ALIAS_CACHE_PATH = OUT_DIR / 'gene_alias_cache.json'

# Load persisted cache from previous runs
alias_cache = json.loads(ALIAS_CACHE_PATH.read_text()) if ALIAS_CACHE_PATH.exists() else {}

# Collect every expanded token that:
#   - isn't a direct var_name match in any adata
#   - isn't a pure-letter family prefix (CCL, CXCL …) — those are handled by prefix expansion
#   - hasn't been cached yet
all_var_names = set()
for adata in adatas.values():
    all_var_names.update(adata.var_names)

need_lookup = set()
for genes in gene_sets.values():
    for gene_raw in genes:
        for gene in expand_gene_range(gene_raw):
            if (
                gene not in all_var_names
                and not is_family_prefix(gene)   # skip family prefixes
                and gene not in alias_cache
            ):
                need_lookup.add(gene)

print(f'Tokens queued for mygene alias lookup: {len(need_lookup)}')
if need_lookup:
    print(' ', sorted(need_lookup))

if need_lookup:
    mg = mygene.MyGeneInfo()
    results = mg.querymany(
        list(need_lookup),
        scopes='alias,symbol,name',
        fields='symbol',
        species='human',
        returnall=False,
        verbose=False,
    )
    n_resolved = 0
    for r in results:
        q = r.get('query', '')
        if not q:
            continue
        if not r.get('notfound', False) and 'symbol' in r:
            alias_cache[q] = r['symbol']
            n_resolved += 1
        else:
            alias_cache[q] = None  # cache miss so we don't re-query next run

    ALIAS_CACHE_PATH.write_text(json.dumps(alias_cache, indent=2))
    print(f'\nResolved {n_resolved}/{len(need_lookup)} via mygene')
    resolved_map = {k: v for k, v in alias_cache.items() if k in need_lookup and v}
    print('Alias mappings found:')
    for alias, canonical in sorted(resolved_map.items()):
        print(f'  {alias!r:25s} → {canonical!r}')
else:
    print('Nothing to query — all tokens are direct matches, family prefixes, or already cached')

Tokens queued for mygene alias lookup: 0
Nothing to query — all tokens are direct matches, family prefixes, or already cached


In [19]:
for depot, adata in adatas.items():
    print(f'\n=== {depot} ===')
    ensembl2var, entrez2var = build_var_lookups(adata)
    print(f'  var lookups: {len(ensembl2var)} Ensembl, {len(entrez2var)} Entrez')

    resolution_log = []  # collect per-set resolution stats for summary

    for gs_name, genes in gene_sets.items():
        col = score_col_map[gs_name]
        valid, counts = resolve_gene_list(genes, adata, ensembl2var, entrez2var)

        resolution_log.append({
            'gene_set': gs_name,
            'n_input':   len(genes),
            'n_found':   len(valid),
            'by_symbol': counts['symbol'],
            'by_ensembl': counts['ensembl'],
            'by_entrez':  counts['entrez'],
            'missing':    counts['missing'],
        })

        if len(valid) == 0:
            adata.obs[col] = np.nan
            continue

        sc.tl.score_genes(
            adata,
            gene_list=valid,
            score_name=col,
            ctrl_size=max(50, len(valid) * 10),
            use_raw=False,
        )

    # Print resolution summary table
    log_df = pd.DataFrame(resolution_log).set_index('gene_set')
    print(log_df.to_string())
    n_scored = sum(1 for c in score_col_map.values() if c in adata.obs.columns and adata.obs[c].notna().any())
    print(f'\n  Scored: {n_scored}/{len(gene_sets)} gene sets')


=== omentum ===
  var lookups: 26158 Ensembl, 19368 Entrez
                                                                                         n_input  n_found  by_symbol  by_ensembl  by_entrez  missing
gene_set                                                                                                                                            
KEGG_ECM_RECEPTOR_INTERACTION                                                                 84       83         83           0          0        1
KEGG_CELL_ADHESION_MOLECULES_CAMS                                                            133      127        127           0          0        6
Mesencyhmal                                                                                    3        3          2           0          0        0
Adipogenic                                                                                    11       10         10           0          0        0
Fibrosis                                      

In [20]:
# Verify: show per-depot score coverage
for depot, adata in adatas.items():
    cols_present = [c for c in score_col_map.values() if c in adata.obs.columns]
    print(f'{depot}: {len(cols_present)}/{len(score_col_map)} gene sets scored')

omentum: 36/36 gene sets scored
subq: 36/36 gene sets scored


## 4. Violin Plots

Each figure shows one gene set across both depots.  
Rows = depot, columns = cell types (coarse `cell_type_`), hue = cohort or sex.  
Saved as `results/gene_sets/figures/violin/<gene_set>_cohort.pdf` etc.

In [37]:
def violin_by_group(
    adatas: dict,
    gs_name: str,
    group_col: str,
    palette: dict,
    ct_col: str = 'cell_type_',
    save_path=None,
):
    """
    Violin plot: x=cell type, hue=group_col, rows=depot.
    Only cell types with ≥20 cells in any group are shown.
    """
    score_col = score_col_map[gs_name]
    depots    = list(adatas.keys())

    fig, axes = plt.subplots(
        len(depots), 1,
        figsize=(max(12, len(adatas[depots[0]].obs[ct_col].unique()) * 1.4), 4 * len(depots)),
        squeeze=False,
    )

    for row_i, depot in enumerate(depots):
        ax    = axes[row_i, 0]
        adata = adatas[depot]

        if score_col not in adata.obs.columns:
            ax.set_title(f'{depot} — score not available')
            continue

        plot_df = adata.obs[[ct_col, group_col, score_col]].copy().dropna()

        # Keep cell types with ≥20 cells total
        ct_counts = plot_df[ct_col].value_counts()
        keep_cts  = ct_counts[ct_counts >= 20].index.tolist()
        plot_df   = plot_df[plot_df[ct_col].isin(keep_cts)]

        order = sorted(plot_df[ct_col].unique())

        sns.violinplot(
            data=plot_df, x=ct_col, y=score_col,
            hue=group_col, palette=palette,
            order=order, inner='box',
            scale='width', cut=0,
            linewidth=0.8, ax=ax,
        )
        ax.set_title(f'{depot}', fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Score', fontsize=10)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
        ax.axhline(0, lw=0.6, ls='--', color='grey')
        ax.legend(title=group_col, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    fig.suptitle(f'{gs_name}  |  by {group_col}  |  {ct_col}', fontsize=11, y=1.01)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()

In [39]:
# ── Generate all violin plots ──────────────────────────────────────────────────
for gs_name in gene_sets:
    for ct_col in ['cell_type_', 'cell_type2_']:
        ct_label = ct_col.rstrip('_')

        # by cohort
        violin_by_group(
            adatas, gs_name,
            group_col='cohort', palette=COHORT_PALETTE,
            ct_col=ct_col,
            save_path=FIG_DIR / 'violin' / f'{safe_name(gs_name)}_{ct_label}_cohort.pdf',
        )

        # by sex
        violin_by_group(
            adatas, gs_name,
            group_col='sex', palette=SEX_PALETTE,
            ct_col=ct_col,
            save_path=FIG_DIR / 'violin' / f'{safe_name(gs_name)}_{ct_label}_sex.pdf',
        )

print('Violin plots saved to', FIG_DIR / 'violin')

Violin plots saved to /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures/violin


## 5. Overview Heatmaps

One figure per depot × annotation level.  
- **Columns**: each cell type shown as a pair of adjacent columns (Healthy | Unhealthy).  
- **Rows**: gene sets, ordered by hierarchical clustering (correlation distance, average linkage) with a dendrogram on the left.  
- **Values**: mean gene-set score per group, z-scored across all cell-type × cohort combinations for each gene set.  
- Parameters `ct_subset` (ordered list) and `gs_subset` let you slice and reorder.

In [60]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe


def score_heatmap_overview(
    adatas,
    depot: str,
    ct_col: str,
    split_col: str = 'cohort',
    split_order: list = None,
    ct_subset: list = None,
    gs_subset: list = None,
    min_cells: int = 10,
    alpha: float = 0.05,
    save_path=None,
    figsize=None,
    cmap: str = 'RdBu_r',
):
    """
    Overview heatmap: gene sets (rows, clustered with dendrogram)
    × cell types (paired columns per split level).

    Statistical test: Mann-Whitney U (split level 0 vs 1) per gene-set × cell-type.
    FDR corrected (BH) across all tests. Marks: * <0.05  ** <0.01  *** <0.001.

    Parameters
    ----------
    split_col   : obs column to split columns by (e.g. 'cohort' or 'sex')
    split_order : explicit level order; None = sorted alphabetically
    ct_subset   : ordered cell type list (None = all, sorted)
    gs_subset   : gene set list (None = all scored)
    min_cells   : minimum cells per group for mean / test
    alpha       : FDR threshold for * marks
    """
    adata  = adatas[depot]
    groups = list(split_order) if split_order is not None else sorted(
        adata.obs[split_col].dropna().unique().tolist()
    )

    # ── Select gene sets ──────────────────────────────────────────────────────
    gs_names = gs_subset if gs_subset is not None else list(score_col_map.keys())
    gs_names = [g for g in gs_names
                if score_col_map.get(g, '') in adata.obs.columns
                and adata.obs[score_col_map[g]].notna().any()]
    if not gs_names:
        print(f'No scored gene sets for {depot}/{ct_col}'); return

    # ── Select & order cell types ─────────────────────────────────────────────
    all_cts  = adata.obs[ct_col].dropna().unique().tolist()
    ct_order = [ct for ct in ct_subset if ct in all_cts] if ct_subset else sorted(all_cts)
    if not ct_order:
        print(f'No matching cell types for {depot}/{ct_col}'); return

    n_gs     = len(gs_names)
    n_cts    = len(ct_order)
    n_groups = len(groups)
    n_cols   = n_cts * n_groups
    col_tuples = [(ct, grp) for ct in ct_order for grp in groups]

    # ── Mean score matrix (gene_sets × columns) ───────────────────────────────
    data = np.full((n_gs, n_cols), np.nan)
    for gi, gs in enumerate(gs_names):
        sc_col = score_col_map[gs]
        for ci, (ct, grp) in enumerate(col_tuples):
            mask = (adata.obs[ct_col] == ct) & (adata.obs[split_col] == grp)
            vals = adata.obs.loc[mask, sc_col].dropna()
            if len(vals) >= min_cells:
                data[gi, ci] = vals.mean()

    mat = pd.DataFrame(data, index=gs_names)

    # ── Z-score each gene set row across all columns ──────────────────────────
    mat_z = mat.copy()
    for i in range(n_gs):
        row = mat_z.iloc[i].values.astype(float)
        std = np.nanstd(row)
        mu  = np.nanmean(row)
        mat_z.iloc[i] = (row - mu) / std if std > 0 else np.zeros(n_cols)
    mat_z = mat_z.fillna(0)

    # ── Hierarchical clustering of gene sets ──────────────────────────────────
    if n_gs >= 3:
        dist      = np.nan_to_num(pdist(mat_z.values, metric='correlation'), nan=1.0)
        Z_link    = linkage(dist, method='average')
        row_order = dendrogram(Z_link, no_plot=True)['leaves']  # bottom→top
    else:
        Z_link    = None
        row_order = list(range(n_gs))

    mat_plot  = mat_z.iloc[row_order[::-1]]
    gs_labels = mat_plot.index.tolist()

    # ── Mann-Whitney U: groups[0] vs groups[1] per (gene_set, cell_type) ──────
    fdr = np.ones((n_gs, n_cts))
    if n_groups >= 2:
        g0, g1 = groups[0], groups[1]
        pvals_flat = []
        for gi, gs in enumerate(gs_names):
            sc_col = score_col_map[gs]
            for ci, ct in enumerate(ct_order):
                m0 = (adata.obs[ct_col] == ct) & (adata.obs[split_col] == g0)
                m1 = (adata.obs[ct_col] == ct) & (adata.obs[split_col] == g1)
                v0 = adata.obs.loc[m0, sc_col].dropna().values
                v1 = adata.obs.loc[m1, sc_col].dropna().values
                if len(v0) >= min_cells and len(v1) >= min_cells:
                    try:
                        _, p = stats.mannwhitneyu(v0, v1, alternative='two-sided')
                        pvals_flat.append(p)
                    except ValueError:
                        pvals_flat.append(1.0)
                else:
                    pvals_flat.append(1.0)
        _, fdr_flat, _, _ = multipletests(pvals_flat, method='fdr_bh')
        fdr = fdr_flat.reshape(n_gs, n_cts)

    # ── Figure dimensions ─────────────────────────────────────────────────────
    cell_w   = max(0.25, min(0.6,  5.0 / n_cols))
    row_h    = max(0.22, min(0.45, 10.0 / n_gs))
    heat_w   = n_cols * cell_w
    heat_h   = n_gs   * row_h
    dendro_w = max(1.0, heat_w * 0.15)
    cbar_w   = 0.25
    label_h  = 1.5

    max_label_len = max(len(s) for s in gs_names)
    gs_label_w = min(5.0, max(2.0, max_label_len * 0.065))

    if figsize is None:
        figsize = (dendro_w + heat_w + gs_label_w + cbar_w + 0.5,
                   heat_h + label_h + 0.8)

    # ── Layout: 4 columns × 2 rows ────────────────────────────────────────────
    fig   = plt.figure(figsize=figsize)
    gsfig = gridspec.GridSpec(
        2, 4, figure=fig,
        width_ratios  = [dendro_w, heat_w, gs_label_w, cbar_w],
        height_ratios = [label_h,  heat_h],
        hspace=0.01, wspace=0.02,
    )
    ax_lab    = fig.add_subplot(gsfig[0, 1])
    ax_dendro = fig.add_subplot(gsfig[1, 0])
    ax_heat   = fig.add_subplot(gsfig[1, 1])
    ax_gslab  = fig.add_subplot(gsfig[1, 2])
    ax_cbar   = fig.add_subplot(gsfig[1, 3])

    # ── Dendrogram ────────────────────────────────────────────────────────────
    if Z_link is not None:
        dendrogram(
            Z_link, orientation='left', ax=ax_dendro,
            no_labels=True, color_threshold=0,
            above_threshold_color='#666666',
        )
    ax_dendro.axis('off')
    ax_dendro.set_ylim(0, n_gs * 10)

    # ── Heatmap ───────────────────────────────────────────────────────────────
    lim = max(0.1, np.nanpercentile(np.abs(mat_z.values), 98))
    im  = ax_heat.imshow(
        mat_plot.values,
        cmap=cmap, vmin=-lim, vmax=lim,
        aspect='auto', interpolation='nearest', origin='upper',
        extent=[-0.5, n_cols - 0.5, 0, n_gs * 10],
    )
    ax_heat.set_ylim(0, n_gs * 10)
    ax_heat.set_yticks([])
    ax_heat.grid(False)

    ax_heat.set_xticks(range(n_cols))
    ax_heat.set_xticklabels(
        [grp[0] for _, grp in col_tuples],
        fontsize=max(6, min(9, 50 / n_cols)),
    )
    ax_heat.tick_params(axis='x', length=0)

    # Thin white lines between split levels within each cell type
    for i in range(n_cts):
        for j in range(1, n_groups):
            ax_heat.axvline(i * n_groups + j - 0.5, color='white', linewidth=1.0)
    # Thicker white lines between cell-type groups
    for i in range(1, n_cts):
        ax_heat.axvline(i * n_groups - 0.5, color='white', linewidth=3)

    # ── Significance marks (centered over each cell-type pair) ────────────────
    gs_orig_idx = {gs: i for i, gs in enumerate(gs_names)}
    mark_fs     = max(8, min(9, 55 / max(n_gs, n_cts)))
    for k, gs in enumerate(gs_labels):              # k = display row, 0 = top
        gi = gs_orig_idx[gs]
        y  = n_gs * 10 - k * 10 - 5                # row centre y
        for ci in range(n_cts):
            fdr_val = fdr[gi, ci]
            if   fdr_val < 0.001: mark = '* * *'
            elif fdr_val < 0.01:  mark = '* *'
            elif fdr_val < alpha: mark = '*'
            else: continue
            x = ci * n_groups + (n_groups - 1) / 2.0
            ax_heat.text(
                x, y, mark,
                ha='center', va='center', fontsize=mark_fs,
                color='white',
                path_effects=[pe.withStroke(linewidth=1.2, foreground='black')],
            )

    # ── Gene-set name labels ──────────────────────────────────────────────────
    ax_gslab.set_xlim(0, 1)
    ax_gslab.set_ylim(0, n_gs * 10)
    ax_gslab.axis('off')
    fs_gs = max(8, min(9, 70 / n_gs))
    for k, label in enumerate(gs_labels):
        ax_gslab.text(0.04, n_gs * 10 - k * 10 - 5, label,
                      va='center', ha='left', fontsize=fs_gs)

    # ── Cell-type labels above heatmap ────────────────────────────────────────
    ax_lab.set_xlim(-0.5, n_cols - 0.5)
    ax_lab.set_ylim(0, 1)
    ax_lab.axis('off')
    fs_ct = max(7, min(11, 90 / n_cts))
    for i, ct in enumerate(ct_order):
        center = i * n_groups + (n_groups - 1) / 2.0
        ax_lab.text(center, 0.05, ct.replace('_', ' '),
                    ha='center', va='bottom',
                    fontsize=fs_ct, fontweight='bold', rotation=40)
        ax_lab.plot([i * n_groups - 0.4, (i + 1) * n_groups - 0.6],
                    [0.03, 0.03], 'k-', linewidth=0.8)

    # ── Colorbar ──────────────────────────────────────────────────────────────
    cb = plt.colorbar(im, cax=ax_cbar)
    cb.set_label('Z-score', fontsize=8)
    ax_cbar.tick_params(labelsize=7)

    needs_abbrev = [g for g in groups if len(g) > 1]
    abbrev_str   = ('  (' + ', '.join(f'{g[0]} = {g}' for g in needs_abbrev) + ')'
                    if needs_abbrev else '')
    fig.suptitle(
        f'{depot}  ·  {ct_col}  ·  {split_col}{abbrev_str}'
        f'  ·  * FDR<0.05  ** <0.01  *** <0.001  (Mann-Whitney)',
        fontsize=8, y=1.005,
    )

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()


In [53]:
# ── Generate all overview heatmaps ────────────────────────────────────────────
# Pass ct_subset (ordered list) and gs_subset to restrict / reorder axes.
# Examples:
#   ct_subset = ['FAPs', 'Pre_adipocytes', 'Myofibroblasts', 'ECs', 'T_cells']
#   gs_subset = ['HALLMARK_FATTY_ACID_METABOLISM', 'Fibrosis', ...]

for depot in ['omentum', 'subq']:
    for ct_col in ['cell_type_', 'cell_type2_']:
        ct_label = ct_col.rstrip('_')

        # Cohort: Healthy vs Unhealthy
        score_heatmap_overview(
            adatas, depot=depot, ct_col=ct_col,
            split_col='cohort',
            split_order=['Healthy', 'Unhealthy'],
            save_path=FIG_DIR / f'heatmap_overview_{depot}_{ct_label}_cohort.pdf',
        )

        # Sex: F vs M
        score_heatmap_overview(
            adatas, depot=depot, ct_col=ct_col,
            split_col='sex',
            split_order=['F', 'M'],
            save_path=FIG_DIR / f'heatmap_overview_{depot}_{ct_label}_sex.pdf',
        )

print('Overview heatmaps saved to', FIG_DIR)


Overview heatmaps saved to /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures


### Non-immune subset

In [61]:
# ── Generate all overview heatmaps ────────────────────────────────────────────
# Pass ct_subset (ordered list) and gs_subset to restrict / reorder axes.
# Examples:
#   ct_subset = ['FAPs', 'Pre_adipocytes', 'Myofibroblasts', 'ECs', 'T_cells']
#   gs_subset = ['HALLMARK_FATTY_ACID_METABOLISM', 'Fibrosis', ...]


ct_subset = ['FAPs', 'Pre_adipocytes', 'Myofibroblasts', 'LECs', 'VECs']

for depot in ['omentum', 'subq']:
    for ct_col in ['cell_type2_']:
        ct_label = ct_col.rstrip('_')

        # Cohort: Healthy vs Unhealthy
        score_heatmap_overview(
            adatas, depot=depot, ct_col=ct_col,
            split_col='cohort',
            split_order=['Healthy', 'Unhealthy'],
            save_path=FIG_DIR / f'heatmap_overview_nonim_{depot}_{ct_label}_cohort.pdf',
            ct_subset=ct_subset
        )

        # Sex: F vs M
        score_heatmap_overview(
            adatas, depot=depot, ct_col=ct_col,
            split_col='sex',
            split_order=['F', 'M'],
            save_path=FIG_DIR / f'heatmap_overview_nonim_{depot}_{ct_label}_sex.pdf',
            ct_subset=ct_subset
        )

print('Overview heatmaps saved to', FIG_DIR)


Overview heatmaps saved to /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures


## 5. UMAP Plots

Each figure: 2×2 grid (omentum / subq  ×  Healthy / Unhealthy) coloured by gene set score.  
A second figure repeats with sex split.  
Saved as `results/gene_sets/figures/umap/<gene_set>_cohort.pdf`.

In [40]:
def umap_by_group(
    adatas: dict,
    gs_name: str,
    group_col: str,
    save_path=None,
    cmap: str = 'RdBu_r',
):
    """
    2×N grid: rows=depot, cols=unique group levels.
    Cells coloured by gene set score; uses symmetric colour limits.
    """
    score_col = score_col_map[gs_name]
    depots    = list(adatas.keys())

    # Collect all score values to set symmetric colour limits
    all_scores = []
    for adata in adatas.values():
        if score_col in adata.obs.columns:
            all_scores.append(adata.obs[score_col].dropna().values)
    if not all_scores:
        print(f'{gs_name}: no scores found, skipping UMAP')
        return
    all_scores = np.concatenate(all_scores)
    clim = np.nanpercentile(np.abs(all_scores), 98)   # robust symmetric limit

    first_adata = next(iter(adatas.values()))
    groups      = sorted(first_adata.obs[group_col].dropna().unique())
    n_cols      = len(groups)

    fig, axes = plt.subplots(
        len(depots), n_cols,
        figsize=(4.5 * n_cols, 4 * len(depots)),
        squeeze=False,
    )

    for row_i, depot in enumerate(depots):
        adata = adatas[depot]
        if 'X_umap' not in adata.obsm:
            for ax in axes[row_i]:
                ax.set_title(f'{depot}: no UMAP')
            continue
        umap_coords = adata.obsm['X_umap']
        scores      = adata.obs.get(score_col, pd.Series(np.nan, index=adata.obs.index))

        for col_j, grp in enumerate(groups):
            ax   = axes[row_i, col_j]
            mask = adata.obs[group_col] == grp

            # Background: all cells in grey
            ax.scatter(
                umap_coords[:, 0], umap_coords[:, 1],
                c='lightgrey', s=0.3, linewidths=0, rasterized=True,
            )
            # Foreground: group cells coloured by score
            sc_plot = ax.scatter(
                umap_coords[mask, 0], umap_coords[mask, 1],
                c=scores[mask].fillna(0).values,
                cmap=cmap, vmin=-clim, vmax=clim,
                s=0.8, linewidths=0, rasterized=True,
            )
            ax.set_title(f'{depot}  |  {grp}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_aspect('equal')

        plt.colorbar(sc_plot, ax=axes[row_i, -1], shrink=0.6, label='Score')

    fig.suptitle(f'{gs_name}  |  {group_col}', fontsize=11)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=120)
        plt.close(fig)
    else:
        plt.show()

In [41]:
# ── Generate all UMAP plots ────────────────────────────────────────────────────
for gs_name in gene_sets:
    umap_by_group(
        adatas, gs_name,
        group_col='cohort',
        save_path=FIG_DIR / 'umap' / f'{safe_name(gs_name)}_cohort.pdf',
    )
    umap_by_group(
        adatas, gs_name,
        group_col='sex',
        save_path=FIG_DIR / 'umap' / f'{safe_name(gs_name)}_sex.pdf',
    )

print('UMAP plots saved to', FIG_DIR / 'umap')

UMAP plots saved to /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures/umap


## 6. DE-Based Enrichment (per cell-type scores)

Uses the MAST results (`primerid`, `coef`, `Pr(>Chisq)`, `cell_type`, `score`).  
The `score` column is the MAST combined z-score (used as ranking statistic for GSEA).

- **Large sets (≥15 genes):** `gseapy.prerank` per cell type.  
- **Small sets (<15 genes):** for each cell type, compute:
  - `mean_logFC`: mean `coef` across gene set members found in DE
  - `stouffer_z` / `stouffer_p`: Stouffer's combined z-score using MAST z-scores of gene set members  
    (MAST z ≈ `score`; where missing, treated as 0 / excluded)

Results saved as CSVs to `results/gene_sets/de_enrichment/`.

In [62]:
# ── Load DE results ───────────────────────────────────────────────────────────
de_data = {}
for (depot, ct_col), path in DE_FILES.items():
    df = pd.read_csv(path)
    # MAST results have a plain 'cell_type' column; rename to match the key
    if 'cell_type' in df.columns:
        df = df.rename(columns={'cell_type': ct_col})
    de_data[(depot, ct_col)] = df
    print(f'{depot} / {ct_col}: {len(df)} DE results, '
          f'{df[ct_col].nunique()} cell types, '
          f'{df["primerid"].nunique()} genes')

omentum / cell_type_: 48414 DE results, 10 cell types, 9763 genes
omentum / cell_type2_: 123928 DE results, 26 cell types, 11543 genes
subq / cell_type_: 42491 DE results, 8 cell types, 9534 genes
subq / cell_type2_: 72782 DE results, 15 cell types, 10047 genes


In [63]:
# ── Helper: Stouffer combined z ────────────────────────────────────────────────
def stouffer_combined(z_scores):
    """Returns (combined_z, two-tailed p) for a list/array of z-scores."""
    z = np.asarray(z_scores)
    z = z[~np.isnan(z)]
    if len(z) == 0:
        return np.nan, np.nan
    z_comb = z.sum() / np.sqrt(len(z))
    p      = 2 * stats.norm.sf(np.abs(z_comb))
    return float(z_comb), float(p)


# ── GSEA for large gene sets ───────────────────────────────────────────────────
gsea_results = []

for (depot, ct_col), de_df in de_data.items():
    print(f'\nGSEA: {depot} / {ct_col}')
    for cell_type, grp in de_df.groupby(ct_col):
        ranked = (
            grp.dropna(subset=['score'])
               .drop_duplicates('primerid')
               .set_index('primerid')['score']
               .sort_values(ascending=False)
        )
        if len(ranked) < 10:
            continue

        # Filter large_sets to genes present in this ranking
        tested_large = {
            k: [g for g in v if g in ranked.index]
            for k, v in large_sets.items()
        }
        tested_large = {k: v for k, v in tested_large.items() if len(v) >= MIN_GENES_GSEA}

        if not tested_large:
            continue

        try:
            res = gp.prerank(
                rnk=ranked,
                gene_sets=tested_large,
                min_size=1,
                max_size=10000,
                permutation_num=1000,
                seed=42,
                verbose=False,
            )
            res_df = res.res2d.copy()
            res_df['depot']     = depot
            res_df['ct_col']    = ct_col
            res_df['cell_type'] = cell_type
            res_df['method']    = 'GSEA'
            gsea_results.append(res_df)
        except Exception as e:
            print(f'  GSEA failed for {cell_type}: {e}')

if gsea_results:
    gsea_df = pd.concat(gsea_results, ignore_index=True)
    gsea_df.to_csv(OUT_DIR / 'de_enrichment' / 'gsea_large_sets.csv', index=False)
    print(f'\nGSEA done: {len(gsea_df)} results saved.')
else:
    gsea_df = pd.DataFrame()
    print('No GSEA results generated.')

2026-05-19 22:39:15,825 [WARNING] Duplicated values found in preranked stats: 22.62% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



GSEA: omentum / cell_type_


2026-05-19 22:39:16,191 [WARNING] Duplicated values found in preranked stats: 52.80% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:16,971 [WARNING] Duplicated values found in preranked stats: 4.67% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:17,683 [WARNING] Duplicated values found in preranked stats: 40.53% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:18,412 [WARNING] Duplicated values found in preranked stats: 3.10% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:19,532 [WARNING] Duplicated values found in preranked stats: 17.22% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:20,217 [WARNING] Duplicated values found in preranked stats: 63.13% of genes
The order of those genes wil


GSEA: omentum / cell_type2_


2026-05-19 22:39:22,250 [WARNING] Duplicated values found in preranked stats: 18.83% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:22,476 [WARNING] Duplicated values found in preranked stats: 18.64% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:22,679 [WARNING] Duplicated values found in preranked stats: 17.26% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:22,899 [WARNING] Duplicated values found in preranked stats: 78.45% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:23,117 [WARNING] Duplicated values found in preranked stats: 49.19% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:24,044 [WARNING] Duplicated values found in preranked stats: 44.98% of genes
The order of those genes w


GSEA: subq / cell_type_


2026-05-19 22:39:36,427 [WARNING] Duplicated values found in preranked stats: 65.94% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:37,228 [WARNING] Duplicated values found in preranked stats: 20.82% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:37,995 [WARNING] Duplicated values found in preranked stats: 30.42% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:38,716 [WARNING] Duplicated values found in preranked stats: 64.31% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:39,394 [WARNING] Duplicated values found in preranked stats: 65.33% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:39,695 [WARNING] Duplicated values found in preranked stats: 72.44% of genes
The order of those genes w


GSEA: subq / cell_type2_


2026-05-19 22:39:41,457 [WARNING] Duplicated values found in preranked stats: 46.71% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:41,680 [WARNING] Duplicated values found in preranked stats: 81.14% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:41,942 [WARNING] Duplicated values found in preranked stats: 20.82% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:42,808 [WARNING] Duplicated values found in preranked stats: 72.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:43,026 [WARNING] Duplicated values found in preranked stats: 67.52% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-19 22:39:43,762 [WARNING] Duplicated values found in preranked stats: 84.76% of genes
The order of those genes w


GSEA done: 930 results saved.


In [64]:
# ── Stouffer / mean logFC for small gene sets ──────────────────────────────────
stouffer_results = []

for (depot, ct_col), de_df in de_data.items():
    for cell_type, grp in de_df.groupby(ct_col):
        grp = grp.drop_duplicates('primerid').set_index('primerid')

        for gs_name, genes in small_sets.items():
            overlap = [g for g in genes if g in grp.index]
            if len(overlap) == 0:
                continue

            sub = grp.loc[overlap]
            z_scores = sub['score'].dropna().values  # MAST combined z
            log_fcs  = sub['coef'].dropna().values

            z_comb, p_stouffer = stouffer_combined(z_scores)
            mean_logfc         = float(np.mean(log_fcs)) if len(log_fcs) > 0 else np.nan

            stouffer_results.append({
                'depot':         depot,
                'ct_col':        ct_col,
                'cell_type':     cell_type,
                'gene_set':      gs_name,
                'n_genes_set':   len(genes),
                'n_genes_found': len(overlap),
                'mean_logFC':    mean_logfc,
                'stouffer_z':    z_comb,
                'stouffer_p':    p_stouffer,
                'method':        'Stouffer',
            })

stouffer_df = pd.DataFrame(stouffer_results)

if not stouffer_df.empty:
    # FDR correction within each (depot, ct_col, gene_set) group across cell types
    fdr_chunks = []
    for _, grp in stouffer_df.groupby(['depot', 'ct_col', 'gene_set']):
        pvals = grp['stouffer_p'].fillna(1).values
        _, fdrs, _, _ = multipletests(pvals, method='fdr_bh')
        fdr_chunks.append(grp.assign(stouffer_fdr=fdrs))
    stouffer_df = pd.concat(fdr_chunks, ignore_index=True)
    stouffer_df.to_csv(OUT_DIR / 'de_enrichment' / 'stouffer_small_sets.csv', index=False)
    print(f'Stouffer results: {len(stouffer_df)} rows saved.')
else:
    print('No small sets found in DE results.')

Stouffer results: 459 rows saved.


## 7. Summary Heatmaps

### 7a. GSEA NES heatmap (large gene sets)

In [102]:
from collections import Counter

_DB_TAG = {
    'HALLMARK_': 'Hallmark',
    'KEGG_':     'KEGG',
    'REACTOME_': 'Reactome',
    'GO_':       'GO',
    'WP_':       'WP',
}

def _clean_gs_name(name, with_tag=False):
    """Strip database prefix and prettify a single gene set name.

    with_tag=True always appends (DB); use _make_gs_rename for batch
    collision-aware renaming instead.
    """
    for prefix, tag in _DB_TAG.items():
        if name.startswith(prefix):
            base = name[len(prefix):].replace('_', ' ').title()
            return f'{base} ({tag})' if with_tag else base
    return name.replace('_', ' ').title()

def _make_gs_rename(names):
    """Return {original_name: display_name} for a collection of gene set names.

    The (DB) suffix is added only where two or more names would otherwise
    collapse to the same display string, e.g.:
        KEGG_OXIDATIVE_PHOSPHORYLATION     ->  Oxidative Phosphorylation (KEGG)
        HALLMARK_OXIDATIVE_PHOSPHORYLATION ->  Oxidative Phosphorylation (Hallmark)
    Names that are unique after cleaning get no suffix.
    """
    bases     = {n: _clean_gs_name(n) for n in names}
    colliding = {base for base, cnt in Counter(bases.values()).items() if cnt > 1}
    return {
        n: (_clean_gs_name(n, with_tag=True) if bases[n] in colliding else bases[n])
        for n in names
    }


def gsea_heatmap(gsea_df, depot, ct_col, fdr_col='FDR q-val', nes_col='NES',
                 save_path=None, ct_group=None, vlim=None):
    """
    NES heatmap for one depot/annotation level.

    Parameters
    ----------
    vlim : float or None
        Symmetric colour limit (|NES| <= vlim).  Pass the same value for both
        depots to share the colourbar scale.  None = auto from this depot's data.
    ct_group : list or None
        Ordered cell-type subset to display.
    """
    if ct_group is None:
        sub = gsea_df[(gsea_df['depot'] == depot) & (gsea_df['ct_col'] == ct_col)].copy()
    else:
        sub = gsea_df[(gsea_df['depot'] == depot) & (gsea_df['ct_col'] == ct_col)
                      & gsea_df['cell_type'].isin(ct_group)].copy()
        sub['cell_type'] = pd.Categorical(sub['cell_type'], categories=ct_group, ordered=True)

    if sub.empty:
        print(f'No GSEA results for {depot} / {ct_col}')
        return

    # Pivot to cell_type x gene_set NES matrix
    pivot = sub.pivot_table(index='cell_type', columns='Term', values=nes_col, aggfunc='first')
    sig   = sub.pivot_table(index='cell_type', columns='Term', values=fdr_col,  aggfunc='first')

    # Prettify gene-set column names (collision-aware)
    rename = _make_gs_rename(pivot.columns)
    pivot  = pivot.rename(columns=rename)
    sig    = sig.rename(columns=rename)

    mat = pivot.fillna(0)
    lim = vlim if vlim is not None else np.nanpercentile(np.abs(mat.values), 99)

    fig, ax = plt.subplots(
        figsize=(max(6, mat.shape[1] * 0.2 + 2), max(4, mat.shape[0] * 0.6 + 1))
    )

    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')

    # Significance asterisks
    for ri, ct in enumerate(mat.index):
        for ci, gs in enumerate(mat.columns):
            fdr_val = sig.at[ct, gs] if (ct in sig.index and gs in sig.columns) else np.nan
            if not np.isnan(fdr_val) and fdr_val < 0.05:
                ax.text(ci, ri, '*', ha='center', va='center', fontsize=9, color='black')

    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index, fontsize=8)
    plt.colorbar(im, ax=ax, label='NES')
    ax.set_title(f'GSEA NES  |  {depot}  |  {ct_col}  (* = FDR < 0.05)', fontsize=10)
    ax.grid(False)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()


# -- Example: shared colourbar across depots ----------------------------------
# for ct_col in ['cell_type_', 'cell_type2_']:
#     shared_lim = np.nanpercentile(
#         gsea_df[gsea_df['ct_col'] == ct_col]['NES'].dropna().abs(), 99
#     )
#     for depot in ['omentum', 'subq']:
#         gsea_heatmap(
#             gsea_df, depot, ct_col, vlim=shared_lim,
#             save_path=FIG_DIR / f'gsea_NES_{depot}_{ct_col.rstrip("_")}.pdf',
#         )


In [103]:
ct_group = ["FAPs", "Pre_adipocytes", "VECs", "LECs", "Myofibroblasts"]

vlim = gsea_df.loc[gsea_df.cell_type.isin(ct_group) & (gsea_df["ct_col"] == "cell_type2_"), 'NES'].abs().max()

for depot in ['omentum', 'subq']:
    for ct_col in ['cell_type2_']:
        gsea_heatmap(
            gsea_df, depot, ct_col,
            save_path=FIG_DIR / f'gsea_NES_{depot}_{ct_col.rstrip("_")}_nonim.pdf',
            ct_group=ct_group,
            vlim=vlim
        )

### 7b. Stouffer Z heatmap (small gene sets)

In [90]:
def stouffer_heatmap(stouffer_df, depot, ct_col, save_path=None, ct_group=None):
    if ct_group is None:
        sub = stouffer_df[(stouffer_df['depot'] == depot) & (stouffer_df['ct_col'] == ct_col)].copy()
    else:
        sub = stouffer_df[(stouffer_df['depot'] == depot) & (stouffer_df['ct_col'] == ct_col) & stouffer_df["cell_type"].isin(ct_group)].copy()
        sub["cell_type"] = pd.Categorical(sub["cell_type"], categories=ct_group, ordered=True)
    if sub.empty:
        return

    pivot_z   = sub.pivot_table(index='cell_type', columns='gene_set', values='stouffer_z',   aggfunc='first')
    pivot_fdr = sub.pivot_table(index='cell_type', columns='gene_set', values='stouffer_fdr', aggfunc='first')

    fig, ax = plt.subplots(
        figsize=(max(4, pivot_z.shape[1] * 1.2 + 1), max(3, pivot_z.shape[0] * 0.5 + 1))
    )
    mat = pivot_z.fillna(0)
    lim = max(1, np.nanpercentile(np.abs(mat.values), 99))

    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')

    for ri, ct in enumerate(mat.index):
        for ci, gs in enumerate(mat.columns):
            fdr_val = pivot_fdr.at[ct, gs] if (ct in pivot_fdr.index and gs in pivot_fdr.columns) else np.nan
            if not np.isnan(fdr_val) and fdr_val < 0.05:
                ax.text(ci, ri, '*', ha='center', va='center', fontsize=9, color='black')

    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index, fontsize=9)
    plt.colorbar(im, ax=ax, label='Stouffer Z')
    ax.set_title(f'Stouffer Z  |  {depot}  |  {ct_col}  (* = FDR < 0.05)', fontsize=10)
    ax.grid(False)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()


# if not stouffer_df.empty:
#     for depot in ['omentum', 'subq']:
#         for ct_col in ['cell_type_', 'cell_type2_']:
#             stouffer_heatmap(
#                 stouffer_df, depot, ct_col,
#                 save_path=FIG_DIR / f'stouffer_Z_{depot}_{ct_col.rstrip("_")}.pdf',
#             )

In [91]:
ct_group = ["FAPs", "Pre_adipocytes", "VECs", "LECs", "Myofibroblasts"]

for depot in ['omentum', 'subq']:
    for ct_col in ['cell_type2_']:
        stouffer_heatmap(
            stouffer_df, depot, ct_col,
            save_path=FIG_DIR / f'stouffer_Z_{depot}_{ct_col.rstrip("_")}_nonim.pdf',
            ct_group=ct_group
        )

## 8. (Optional) Interactive inline preview

Run any of the cells below to display a plot inline without saving — useful for spot-checking.

In [ ]:
# Example: show violin for one gene set inline
violin_by_group(
    adatas,
    gs_name='HALLMARK_FATTY_ACID_METABOLISM',   # change to any gene set name
    group_col='cohort',
    palette=COHORT_PALETTE,
    ct_col='cell_type_',
    save_path=None,   # None = display inline
)

In [ ]:
# Example: show UMAP for one gene set inline
umap_by_group(
    adatas,
    gs_name='HALLMARK_OXIDATIVE_PHOSPHORYLATION',   # change to any gene set name
    group_col='cohort',
    save_path=None,
)

In [ ]:
# Example: show Stouffer heatmap inline
if not stouffer_df.empty:
    stouffer_heatmap(stouffer_df, depot='omentum', ct_col='cell_type_', save_path=None)

In [ ]:
# Example: show GSEA heatmap inline
if not gsea_df.empty:
    gsea_heatmap(gsea_df, depot='omentum', ct_col='cell_type_', save_path=None)

## 9. Publication Figures

### Figure 1 — Module scores: FAPs & Pre-adipocytes (Mesenchymal / Fibrosis / Adipogenic)
### Figure 2 — EC log expression: individual Lipid gene set genes

In [69]:
# ─── Shared setup for publication figures ─────────────────────────────────────
import scipy.sparse

COHORT_RENAME = {'Healthy': 'Ob', 'Unhealthy': 'Ob+T2D'}
PUB_PALETTE   = {'Ob': '#4393c3', 'Ob+T2D': '#d6604d'}
DEPOT_LABELS  = {'omentum': 'Omentum', 'subq': 'Subcutaneous'}

plt.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.labelsize':     10,
    'axes.titlesize':     11,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     0.8,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'pdf.fonttype':       42,   # embed TrueType fonts in PDF
    'ps.fonttype':        42,
})

def _sig(p):
    """Return significance asterisks or None."""
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else None

def _bracket(ax, x_c, y_base, y_range, p, lw=0.8, fs=8, half_w=0.18):
    """Draw a significance bracket centred at x_c placed at y_base."""
    mark = _sig(p)
    if mark is None:
        return
    h  = 0.04 * abs(y_range)
    xl, xr = x_c - half_w, x_c + half_w
    ax.plot([xl, xl, xr, xr], [y_base, y_base + h, y_base + h, y_base],
            lw=lw, color='k', clip_on=False)
    ax.text(x_c, y_base + h + 0.008 * abs(y_range), mark,
            ha='center', va='bottom', fontsize=fs, clip_on=False)

print('Shared publication-figure setup complete.')

Shared publication-figure setup complete.


In [70]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 1: Module scores — FAPs & Pre-adipocytes
# Rows = depot (Omentum top, Subcutaneous bottom)
# Columns = cell type (FAPs left, Pre-adipocytes right)
# x-axis = gene set | hue = cohort (Ob vs Ob+T2D)
# ─────────────────────────────────────────────────────────────────────────────

FIG1_GS    = ['Mesencyhmal', 'Fibrosis', 'Adipogenic']
GS_LABELS  = {'Mesencyhmal': 'Mesenchymal', 'Fibrosis': 'Fibrosis', 'Adipogenic': 'Adipogenic'}
GS_ORDER   = [GS_LABELS[g] for g in FIG1_GS]
CT_LIST    = ['FAPs', 'Pre_adipocytes']
CT_LABELS  = {'FAPs': 'FAPs', 'Pre_adipocytes': 'Pre-adipocytes'}

# ── Collect module scores in long format ──────────────────────────────────────
records = []
for depot, adata in adatas.items():
    obs = adata.obs.copy()
    obs['cohort_rn'] = obs['cohort'].map(COHORT_RENAME)
    for ct in CT_LIST:
        mask = obs['cell_type_'] == ct
        if not mask.any():
            continue
        sub_obs = obs[mask]
        for gs in FIG1_GS:
            col = score_col_map[gs]
            if col not in sub_obs.columns:
                continue
            records.append(pd.DataFrame({
                'score':     sub_obs[col].values,
                'cohort':    sub_obs['cohort_rn'].values,
                'depot':     depot,
                'cell_type': ct,
                'gene_set':  GS_LABELS[gs],
            }))

score_df = pd.concat(records, ignore_index=True).dropna(subset=['score'])

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(8.0, 7.0))

for row_i, depot in enumerate(['omentum', 'subq']):
    for col_j, ct in enumerate(CT_LIST):
        ax  = axes[row_i, col_j]
        sub = score_df[(score_df['depot'] == depot) & (score_df['cell_type'] == ct)]

        if sub.empty:
            ax.set_visible(False)
            continue

        sns.violinplot(
            data=sub, x='gene_set', y='score',
            hue='cohort', hue_order=['Ob', 'Ob+T2D'],
            order=GS_ORDER, palette=PUB_PALETTE,
            inner='box', scale='width', cut=0,
            linewidth=0.8, ax=ax,
        )

        # ── Significance brackets ─────────────────────────────────────────────
        vals  = sub['score'].dropna().values
        y_lo  = np.nanpercentile(vals, 0.5)
        y_hi  = np.nanpercentile(vals, 99.5)
        y_rng = y_hi - y_lo
        y_brk = y_hi + 0.08 * y_rng

        for gi, gs_lbl in enumerate(GS_ORDER):
            d0 = sub[(sub['gene_set'] == gs_lbl) & (sub['cohort'] == 'Ob')]['score'].dropna()
            d1 = sub[(sub['gene_set'] == gs_lbl) & (sub['cohort'] == 'Ob+T2D')]['score'].dropna()
            if len(d0) >= 5 and len(d1) >= 5:
                _, p = stats.mannwhitneyu(d0, d1, alternative='two-sided')
                _bracket(ax, gi, y_brk, y_rng, p)

        ax.set_ylim(y_lo - 0.04 * y_rng, y_brk + 0.18 * y_rng)

        # ── Axes style ────────────────────────────────────────────────────────
        ax.axhline(0, lw=0.6, ls='--', color='#aaaaaa', zorder=0)
        ax.set_title(f'{DEPOT_LABELS[depot]}  ·  {CT_LABELS[ct]}',
                     fontsize=11, fontweight='bold', pad=6)
        ax.set_xlabel('')
        ax.set_ylabel('Module score' if col_j == 0 else '', fontsize=10)

        leg = ax.get_legend()
        if row_i == 0 and col_j == 1:
            leg.set_title('Cohort')
            for t in leg.get_texts(): t.set_fontsize(9)
            leg.get_frame().set_visible(False)
        elif leg:
            leg.remove()

sns.despine(fig=fig, trim=False)
fig.suptitle('Module scores in stromal cell types', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.93, right=0.87)

save_path = FIG_DIR / 'pub_stromal_module_scores.pdf'
fig.savefig(save_path, bbox_inches='tight', dpi=300)
plt.close()
print(f'Saved → {save_path}')

Saved → /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures/pub_stromal_module_scores.pdf


In [107]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 2: EC log-normalised expression — individual Lipid gene set genes
# Rows = depot (Omentum top, Subcutaneous bottom)
# x-axis = gene | hue = cohort (Ob vs Ob+T2D)
# ─────────────────────────────────────────────────────────────────────────────

# # ── Expand gene list (PLIN1-5 → PLIN1 … PLIN5) ───────────────────────────────
# lipid_expanded = []
# for g in gene_sets['Lipid']:
#     lipid_expanded.extend(expand_gene_range(g))
# print('Lipid genes (expanded):', lipid_expanded)

lipid_expanded = """HLA-DQA1
HLA-DRB1
HLA-A
HLA-DPB1
HLA-DQB1
HLA-DPA1
HLA-DRB5
HLA-DRA
CD74""".split("\n")

# ── Extract log-normalised expression for ECs ─────────────────────────────────
rec2 = []
for depot, adata in adatas.items():
    mask = adata.obs['cell_type2_'] == 'VECs'
    if not mask.any():
        print(f'{depot}: no ECs found'); continue
    ec_adata = adata[mask]
    cohorts  = ec_adata.obs['cohort'].map(COHORT_RENAME).values

    for gene in lipid_expanded:
        if gene not in adata.var_names:
            print(f'  {depot}: {gene} not in var_names — skipping')
            continue
        idx   = np.where(adata.var_names == gene)[0][0]
        X_col = ec_adata.X[:, idx]
        expr  = (np.asarray(X_col.todense()).flatten()
                 if scipy.sparse.issparse(X_col)
                 else np.asarray(X_col).flatten())
        for val, coh in zip(expr, cohorts):
            rec2.append({'depot': depot, 'gene': gene,
                         'expression': float(val), 'cohort': coh})

ec_df = pd.DataFrame(rec2)
print(f'\nEC Lipid expression: {len(ec_df):,} rows | '
      f'depots: {ec_df["depot"].unique().tolist()} | '
      f'genes: {ec_df["gene"].unique().tolist()}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10.0, 7.0))

for row_i, depot in enumerate(['omentum', 'subq']):
    ax  = axes[row_i]
    sub = ec_df[ec_df['depot'] == depot]
    if sub.empty:
        ax.set_visible(False); continue

    gene_order = [g for g in lipid_expanded if g in sub['gene'].unique()]

    sns.violinplot(
        data=sub, x='gene', y='expression',
        hue='cohort', hue_order=['Ob', 'Ob+T2D'],
        order=gene_order, palette=PUB_PALETTE,
        inner='box', scale='width', cut=0,
        linewidth=0.8, ax=ax,
    )

    # ── Significance brackets ─────────────────────────────────────────────────
    vals  = sub['expression'].dropna().values
    y_lo  = np.nanpercentile(vals, 0.5)
    y_hi  = np.nanpercentile(vals, 99.5)
    y_rng = y_hi - y_lo
    y_brk = y_hi + 0.08 * y_rng

    for gi, gene in enumerate(gene_order):
        d0 = sub[(sub['gene'] == gene) & (sub['cohort'] == 'Ob')]['expression'].dropna()
        d1 = sub[(sub['gene'] == gene) & (sub['cohort'] == 'Ob+T2D')]['expression'].dropna()
        if len(d0) >= 5 and len(d1) >= 5:
            _, p = stats.mannwhitneyu(d0, d1, alternative='two-sided')
            _bracket(ax, gi, y_brk, y_rng, p)

    ax.set_ylim(y_lo - 0.04 * y_rng, y_brk + 0.18 * y_rng)

    ax.axhline(0, lw=0.5, ls='--', color='#aaaaaa', zorder=0)
    ax.set_title(f'{DEPOT_LABELS[depot]}  ·  VECs  ·  Lipid gene set',
                 fontsize=11, fontweight='bold', pad=6)
    ax.set_xlabel('Gene', fontsize=10)
    ax.set_ylabel('Log-normalised expression', fontsize=10)

    leg = ax.get_legend()
    if row_i == 0:
        leg.set_title('Cohort')
        for t in leg.get_texts(): t.set_fontsize(9)
        leg.get_frame().set_visible(False)
    elif leg:
        leg.remove()

sns.despine(fig=fig, trim=False)
fig.suptitle('Lipid gene expression in endothelial cells', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.93, right=0.87)

save_path = FIG_DIR / 'pub_EC_lipid_gene_expression.pdf'
fig.savefig(save_path, bbox_inches='tight', dpi=300)
plt.close()
print(f'Saved → {save_path}')


EC Lipid expression: 11,304 rows | depots: ['omentum', 'subq'] | genes: ['HLA-DQA1', 'HLA-DRB1', 'HLA-A', 'HLA-DPB1', 'HLA-DQB1', 'HLA-DPA1', 'HLA-DRB5', 'HLA-DRA', 'CD74']
Saved → /Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/gene_sets/figures/pub_EC_lipid_gene_expression.pdf
